# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library, referencing all dataset entities by their `@id` fields for consistency and reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This step accesses the Croissant schema and creates a metadata object.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"License: {metadata.license}")
print(f"Version: {metadata.version}")

## 2. Data Overview
Review available record sets, fields, and their IDs (`@id`).

Below, you can examine the schema structure, including record sets and their fields. All are referenced by `@id` as required for Croissant datasets.

In [ ]:
# List all record sets and their fields by their @id
record_sets = list(metadata.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"Record set name: {rs.name} (ID: {rs['@id']})")
    print("  Fields:")
    for field in rs.fields:
        print(f"    {field.name} (ID: {field['@id']}) - type: {field.data_type}")
    print()

## 3. Data Extraction
Load data from one or multiple record sets into DataFrames for analysis.

We'll use the record set and field `@id`s as listed above. If the dataset defines several record sets, they'll each be loaded and referenced by `@id` for consistent, reproducible downstream processing.

In [ ]:
# Gather the @id of each record set
record_set_ids = [rs['@id'] for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded data for record set {record_set_id} - shape: {df.shape}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(2))
    else:
        print(f"No records found for record set {record_set_id}")

## 4. Exploratory Data Analysis (EDA)
We will select an appropriate numeric field (column) by its `@id` from one of the loaded record sets and perform basic processing: filtering, normalization, and grouping. Make sure to update the `numeric_field_id` and `group_field_id` variables to match a field from your dataset (see above for field `@id`s and types).

**Referencing by `@id` is required.** If you're unsure which field is numeric, revisit the previous cell for types.

In [ ]:
# --- Update these IDs to match a field and record set from the dataset ---
record_set_id = record_set_ids[0] if record_set_ids else None

# For demonstration, fetch the first numeric field found
df = dataframes.get(record_set_id)
numeric_field_id = None
group_field_id = None
if df is not None:
    # Using the metadata, find the numeric field in the record set
    record_set_obj = next(rs for rs in metadata.record_sets if rs['@id'] == record_set_id)
    for field in record_set_obj.fields:
        # Typical numeric types; includes handling for ambiguous cases
        if str(field.data_type).lower() in ('number', 'float', 'integer', 'double'):
            if field['@id'] in df.columns:
                numeric_field_id = field['@id']
                break
    # Fallback: try standard names
    if not numeric_field_id:
        for col in df.columns:
            if df[col].dtype.kind in 'fi':
                numeric_field_id = col
                break

    # Select a group field (string or categorical type)
    for field in record_set_obj.fields:
        if field['@id'] != numeric_field_id and str(field.data_type).lower() in ('string', 'text', 'category'):
            if field['@id'] in df.columns:
                group_field_id = field['@id']
                break

    print(f"Analyzing record set '{record_set_id}' with numeric field '{numeric_field_id}' and group field '{group_field_id}'\n")
    
    # Filter records: drop NAs, numeric-only
    filtered_df = df.copy()
    if numeric_field_id:
        filtered_df = filtered_df.dropna(subset=[numeric_field_id])
        filtered_df = filtered_df[pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').notnull()]
        filtered_df[numeric_field_id] = filtered_df[numeric_field_id].astype(float)
        threshold = filtered_df[numeric_field_id].mean()
        filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
        print(f"Number of records above threshold (mean={threshold:.2f}): {len(filtered_df)}\n")

        # Normalize numeric field (standard z-score)
        mean = filtered_df[numeric_field_id].mean()
        std = filtered_df[numeric_field_id].std()
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - mean) / std
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Group by group_field if available
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"\nGrouped means by '{group_field_id}':\n{grouped_df.head()}")
else:
    print("No DataFrame loaded to analyze.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Adjust the fields in the plot as appropriate for your loaded data and analysis above (must always reference by `@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].astype(float), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated loading, exploring, analyzing, and visualizing the dataset `Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells` directly from its Croissant schema.

Key actions:
- All references to data and schema entities were handled by their Croissant `@id` fields.
- Record sets and fields were programmatically discovered and explored.
- Basic filtering, normalization, and aggregation was demonstrated using field IDs.

You can continue this exploration by selecting specific fields of analytical/biological interest and chaining more advanced workflows such as machine learning, curation, or export.

For production or downstream FAIR workflows, always refer to programmatic IDs (`@id`) and the latest schema.